# 📋 Revela — Notebook 01 Inspection Report

## What's in the Document

10 sections covering everything from the notebook output:

| Section | Content |
|---|---|
| **Sections 1–3** | Purpose, dataset structure, and the skin tone distribution finding *(the 7.6x gap between FST II and FST VI — your demo's most powerful slide)* |
| **Section 4** | All 4 target conditions with exact sub-label breakdowns and image counts: Melanoma: 490 · Nevus: 485 · Eczema: 1,131 — all viable for training |
| **Sections 5–6** | Data quality assessment *(zero duplicates, perfect image-CSV match)* and clinical structure explanation |
| **Section 7** | Requirements check table mapping every Revela scoping decision to what the data actually supports — 6 ✅ green, 1 ⚠️ amber *(FST VI thinness)* |
| **Sections 8–9** | Risk register and the roadmap for notebooks 02–06 |
| **Section 10** | Exact markdown cell to paste as the final cell in `01_inspect_FP.ipynb` |

---

## ⚠️ Your One Amber Flag to Track

> **FST VI has only 635 images across all 114 conditions.**
>
> When you slice that by condition *(e.g. melanoma on FST VI skin)*, you may have
> fewer than **50 images**. That is too few for reliable accuracy estimates —
> which means **Guardrail E warnings will carry uncertainty that needs to be
> disclosed.**


**Block 1 — Load and Basic Info**

In [1]:
import pandas as pd

df = pd.read_csv('../data/fitzpatrick17k.csv')

print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

Total rows: 16577

Columns: ['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']

First 3 rows:
                            md5hash  fitzpatrick_scale  fitzpatrick_centaur                            label nine_partition_label three_partition_label   qc                                                                                                          url                                                                                     url_alphanum
0  5e82a45bc5d78bd24ae9202d194423f8                  3                    3  drug induced pigmentary changes         inflammatory        non-neoplastic  NaN  https://www.dermaamin.com/site/images/clinical-pic/m/minocycline-pigmentation/minocycline-pigmentation1.jpg  httpwwwdermaamincomsiteimagesclinicalpicmminocyclinepigmentationminocyclinepigmentation1jpg.jpg
1  fa2911a9b13b6f8af79cb700937cc14f                  1                    1             

**Block 2 — Column Types and Missing Values**

In [2]:
print("=== COLUMN TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== UNKNOWN SKIN TONES (-1) ===")
print(f"Rows with unknown skin tone: {(df['fitzpatrick_scale'] == -1).sum()}")
print(f"Rows with known skin tone:   {(df['fitzpatrick_scale'] != -1).sum()}")

=== COLUMN TYPES ===
md5hash                    str
fitzpatrick_scale        int64
fitzpatrick_centaur      int64
label                      str
nine_partition_label       str
three_partition_label      str
qc                         str
url                        str
url_alphanum               str
dtype: object

=== MISSING VALUES ===
md5hash                      0
fitzpatrick_scale            0
fitzpatrick_centaur          0
label                        0
nine_partition_label         0
three_partition_label        0
qc                       16073
url                         41
url_alphanum                 0
dtype: int64

=== UNKNOWN SKIN TONES (-1) ===
Rows with unknown skin tone: 565
Rows with known skin tone:   16012


What to look for: The qc column will have lots of missing values — that's normal. Focus on whether label, fitzpatrick_scale, and url have missing values.


**Block 3 — Skin Tone Distribution**


In [4]:
print("=== FITZPATRICK SCALE DISTRIBUTION ===")
print(df['fitzpatrick_scale'].value_counts().sort_index())

print("\n=== AS PERCENTAGE ===")
total_known = (df['fitzpatrick_scale'] != -1).sum()
for fst in range(1, 7):
    count = (df['fitzpatrick_scale'] == fst).sum()
    pct = (count / total_known) * 100
    bar = '█' * int(pct / 2)
    print(f"FST {fst}: {count:4d} images ({pct:.1f}%) {bar}")

=== FITZPATRICK SCALE DISTRIBUTION ===
fitzpatrick_scale
-1     565
 1    2947
 2    4808
 3    3308
 4    2781
 5    1533
 6     635
Name: count, dtype: int64

=== AS PERCENTAGE ===
FST 1: 2947 images (18.4%) █████████
FST 2: 4808 images (30.0%) ███████████████
FST 3: 3308 images (20.7%) ██████████
FST 4: 2781 images (17.4%) ████████
FST 5: 1533 images (9.6%) ████
FST 6:  635 images (4.0%) █


What to look for: You should see a clear skew toward FST I–III. This is your product story — document it.

**Block 4 — All Unique Conditions**

In [5]:
print(f"Total unique conditions: {df['label'].nunique()}")
print("\n=== TOP 30 CONDITIONS ===")
print(df['label'].value_counts().head(30))

Total unique conditions: 114

=== TOP 30 CONDITIONS ===
label
psoriasis                      653
squamous cell carcinoma        581
lichen planus                  491
basal cell carcinoma           468
allergic contact dermatitis    430
lupus erythematosus            410
neutrophilic dermatoses        361
sarcoidosis                    349
photodermatoses                348
folliculitis                   342
scabies                        339
acne vulgaris                  335
scleroderma                    309
pityriasis rubra pilaris       278
melanoma                       261
nematode infection             260
erythema multiforme            236
granuloma annulare             211
eczema                         204
drug eruption                  200
pityriasis rosea               193
neurofibromatosis              189
acne                           183
porokeratosis actinic          183
mycosis fungoides              182
actinic keratosis              175
hailey hailey disease       

What to look for: Your 4 targets — melanoma, nevus, eczema — should appear in or near the top 30.


**Block 5 — Check Your 4 Target Conditions Specifically**

In [6]:
targets = ['melanoma', 'nevus', 'eczema', 'dermatitis']

print("=== TARGET CONDITION SEARCH ===\n")
for target in targets:
    matches = df[df['label'].str.contains(
        target, case=False, na=False
    )]
    print(f"'{target}' — {len(matches)} total images")
    print(matches['label'].value_counts().to_string())
    print()

=== TARGET CONDITION SEARCH ===

'melanoma' — 490 total images
label
melanoma                              261
superficial spreading melanoma ssm    118
malignant melanoma                    111

'nevus' — 485 total images
label
nevus sebaceous of jadassohn    95
epidermal nevus                 91
nevocytic nevus                 86
halo nevus                      82
congenital nevus                68
becker nevus                    63

'eczema' — 287 total images
label
eczema                204
dyshidrotic eczema     83

'dermatitis' — 844 total images
label
allergic contact dermatitis     430
seborrheic dermatitis           126
acrodermatitis enteropathica     92
neurodermatitis                  69
factitial dermatitis             66
perioral dermatitis              61



What to look for: All the sub-labels that fall under each condition. For example, melanoma might appear as melanoma, nodular melanoma, acral melanoma — you need all of them for your mapping.

**Block 6 — Check Three Partition Labels**

In [7]:
print("=== THREE PARTITION LABELS ===")
print(df['three_partition_label'].value_counts())

print("\n=== NINE PARTITION LABELS ===")
print(df['nine_partition_label'].value_counts())

=== THREE PARTITION LABELS ===
three_partition_label
non-neoplastic    12080
malignant          2263
benign             2234
Name: count, dtype: int64

=== NINE PARTITION LABELS ===
nine_partition_label
inflammatory                    10886
malignant epidermal              1352
genodermatoses                   1194
benign dermal                    1067
benign epidermal                  931
malignant melanoma                573
benign melanocyte                 236
malignant cutaneous lymphoma      182
malignant dermal                  156
Name: count, dtype: int64


What to look for: benign / malignant / non-neoplastic breakdown — tells you which broad bucket each condition falls into.

**Block 7 — Verify Image Links**

In [8]:
print("=== URL SAMPLE ===")
print(df['url'].dropna().head(5).to_string())

print(f"\nTotal rows with URL: {df['url'].notna().sum()}")
print(f"Total rows missing URL: {df['url'].isna().sum()}")

print("\n=== MD5HASH SAMPLE ===")
print(df['md5hash'].head(5).to_string())
print(f"\nUnique hashes: {df['md5hash'].nunique()}")
print(f"Total rows: {len(df)}")
print(f"Duplicates: {len(df) - df['md5hash'].nunique()}")

=== URL SAMPLE ===
0    https://www.dermaamin.com/site/images/clinical...
1    https://www.dermaamin.com/site/images/clinical...
2    https://www.dermaamin.com/site/images/clinical...
3    https://www.dermaamin.com/site/images/clinical...
4    https://www.dermaamin.com/site/images/clinical...

Total rows with URL: 16536
Total rows missing URL: 41

=== MD5HASH SAMPLE ===
0    5e82a45bc5d78bd24ae9202d194423f8
1    fa2911a9b13b6f8af79cb700937cc14f
2    d2bac3c9e4499032ca8e9b07c7d3bc40
3    0a94359e7eaacd7178e06b2823777789
4    a39ec3b1f22c08a421fa20535e037bba

Unique hashes: 16577
Total rows: 16577
Duplicates: 0


What to look for: The md5hash is your image filename. It should match the filenames in your finalfitz17k folder.


**Block 8 — Verify Images Match CSV**

In [9]:
import os

image_folder = '../data/finalfitz17k'
image_files = os.listdir(image_folder)

print(f"Images in folder: {len(image_files)}")
print(f"Rows in CSV: {len(df)}")

# Sample filenames from folder
print(f"\nSample image filenames from folder:")
for f in image_files[:5]:
    print(f"  {f}")

# Sample md5hash from CSV
print(f"\nSample md5hash from CSV:")
for h in df['md5hash'].head(5):
    print(f"  {h}")

Images in folder: 16577
Rows in CSV: 16577

Sample image filenames from folder:
  8da0ad261c76d6ed8587ad2275664650.jpg
  ae608b81693c2a1b5c2d9c7c471b0a4a.jpg
  0ca870413a13e07dc10b44cb327e1289.jpg
  c027d6925af7aa63d143b0c6da90de6e.jpg
  08feca025dfb53bff8826d9c78c5fde8.jpg

Sample md5hash from CSV:
  5e82a45bc5d78bd24ae9202d194423f8
  fa2911a9b13b6f8af79cb700937cc14f
  d2bac3c9e4499032ca8e9b07c7d3bc40
  0a94359e7eaacd7178e06b2823777789
  a39ec3b1f22c08a421fa20535e037bba
